# Anastomosing Function (AF) — Entropy Analysis

**Workflow**
1. Load `Chennel_Data.xlsx` — each sheet = one year, containing a `Distance` column (radial distance of each channel-crossing point).
2. For each year, group `Distance` by unique value and count occurrences → this count is the number of channels at that distance → build the AF vector `x_year = [AF(d1), AF(d2), ...]` sorted by increasing distance.
3. Compute `r = 0.15 * std(x_year)` for each year.
4. Compute Approximate Entropy (ApEn) and Sample Entropy (SampEn) using `m = 2` (and `m+1 = 3`).
5. Collect Year | ApEn | SampEn into a results table and plot (matches Figure 7a in Sarker et al. 2023).

## Step 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Step 2 — Set file path and list sheets (years)

Edit `FILE_PATH` below to point to your Excel file.

In [2]:
FILE_PATH = r"G:\Cahnnel Complexity Ganges\Chennel_Data.xlsx"

xls = pd.ExcelFile(FILE_PATH)
sheet_names = xls.sheet_names
print("Sheets found (years):", sheet_names)

Sheets found (years): ['2010']


## Step 3 — Function to build the AF vector from a single sheet

Groups the `Distance` column by unique value, counts occurrences, and sorts by increasing distance.

In [3]:
def build_af_vector(file_path, sheet_name, distance_col="Distance"):
    """
    Reads one sheet and returns:
        distances : sorted unique distance values
        af_vector : count of occurrences at each distance (number of channels)
    """
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    counts = df[distance_col].value_counts().sort_index()
    distances = counts.index.to_numpy()
    af_vector = counts.to_numpy()
    return distances, af_vector

## Step 4 — Build AF vectors for every year (sheet)

In [4]:
af_data = {}   # year -> af_vector
d_data  = {}   # year -> distances

for sheet in sheet_names:
    distances, af_vector = build_af_vector(FILE_PATH, sheet)
    d_data[sheet] = distances
    af_data[sheet] = af_vector
    print(f"Year {sheet}: AF vector length = {len(af_vector)}")
    print(f"  x_{sheet} = {af_vector}")

Year 2010: AF vector length = 41
  x_2010 = [1 2 2 2 1 1 2 2 3 4 3 3 3 3 3 2 4 2 2 3 5 4 5 1 2 3 2 3 2 3 3 1 2 2 1 2 9
 8 4 2 1]


## Step 4b — Normalize d and AF(d) (as in Section 3.3 of the paper)

The paper normalizes **d by R** (max radial distance, so d runs 0–1) and **AF(d) by the total number of channel intersections** (so AF(d) behaves like a probability distribution, summing to 1). Entropy is computed on this normalized AF vector, not on the raw counts.

In [5]:
d_norm_data = {}    # year -> normalized distances (d / R)
af_norm_data = {}   # year -> normalized AF vector (AF / total count)

for year in sheet_names:
    distances = d_data[year].astype(float)
    af_vector = af_data[year].astype(float)

    R = distances.max()                 # max radial distance for that year
    total_count = af_vector.sum()       # total number of channel intersections

    d_norm = distances / R
    af_norm = af_vector / total_count

    d_norm_data[year] = d_norm
    af_norm_data[year] = af_norm

    print(f"Year {year}: R={R:.0f} m, total_count={total_count:.0f}")
    print(f"  d_norm = {np.round(d_norm, 3)}")
    print(f"  AF_norm = {np.round(af_norm, 4)}")

Year 2010: R=205000 m, total_count=113
  d_norm = [0.024 0.049 0.073 0.098 0.122 0.146 0.171 0.195 0.22  0.244 0.268 0.293
 0.317 0.341 0.366 0.39  0.415 0.439 0.463 0.488 0.512 0.537 0.561 0.585
 0.61  0.634 0.659 0.683 0.707 0.732 0.756 0.78  0.805 0.829 0.854 0.878
 0.902 0.927 0.951 0.976 1.   ]
  AF_norm = [0.0088 0.0177 0.0177 0.0177 0.0088 0.0088 0.0177 0.0177 0.0265 0.0354
 0.0265 0.0265 0.0265 0.0265 0.0265 0.0177 0.0354 0.0177 0.0177 0.0265
 0.0442 0.0354 0.0442 0.0088 0.0177 0.0265 0.0177 0.0265 0.0177 0.0265
 0.0265 0.0088 0.0177 0.0177 0.0088 0.0177 0.0796 0.0708 0.0354 0.0177
 0.0088]


# How the AF is create

In [14]:
year = 2010
AF = np.concatenate([af_norm_data[year] for year in sheet_names])
N = len(AF)
m = 2
n = m + 1
r = 0.15*np.std(AF)
def form_vectors(series, length):
    N = len(series)
    return np.array([series[i:i + length] for i in range(N - length + 1)])

vecs_m = form_vectors(AF, m)

vec_table = pd.DataFrame({
    "Vector": [f"x{i+1}" for i in range(len(vecs_m))],
    "Values": [list(v) for v in vecs_m]
})
print(f"Number of vectors = N - m + 1 = {len(vecs_m)}")
print("tolerance r =", r)
vec_table


Number of vectors = N - m + 1 = 40
tolerance r = 0.0022092020862376836


,Vector,Values
0,x1,"[0.008849557522123894, 0.017699115044247787]"
1,x2,"[0.017699115044247787, 0.017699115044247787]"
2,x3,"[0.017699115044247787, 0.017699115044247787]"
3,x4,"[0.017699115044247787, 0.008849557522123894]"
4,x5,"[0.008849557522123894, 0.008849557522123894]"
5,x6,"[0.008849557522123894, 0.017699115044247787]"
6,x7,"[0.017699115044247787, 0.017699115044247787]"
7,x8,"[0.017699115044247787, 0.02654867256637168]"
8,x9,"[0.02654867256637168, 0.035398230088495575]"
9,x10,"[0.035398230088495575, 0.02654867256637168]"


In [15]:
def distance_matrix(vecs):
    n_vec = len(vecs)
    D = np.zeros((n_vec, n_vec))
    for i in range(n_vec):
        for j in range(n_vec):
            D[i, j] = np.max(np.abs(vecs[i] - vecs[j]))
    return D

D_m = distance_matrix(vecs_m)

labels_m = [f"x{i+1}" for i in range(len(vecs_m))]
D_m_df = pd.DataFrame(D_m, index=labels_m, columns=labels_m)
D_m_df

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,...,x31,x32,x33,x34,x35,x36,x37,x38,x39,x40
x1,0.000000,0.008850,0.008850,0.008850,0.008850,0.000000,0.008850,0.008850,0.017699,0.026549,...,0.017699,0.000000,0.008850,0.008850,0.000000,0.061947,0.070796,0.061947,0.026549,0.008850
x2,0.008850,0.000000,0.000000,0.008850,0.008850,0.008850,0.000000,0.008850,0.017699,0.017699,...,0.008850,0.008850,0.000000,0.008850,0.008850,0.061947,0.061947,0.053097,0.017699,0.008850
x3,0.008850,0.000000,0.000000,0.008850,0.008850,0.008850,0.000000,0.008850,0.017699,0.017699,...,0.008850,0.008850,0.000000,0.008850,0.008850,0.061947,0.061947,0.053097,0.017699,0.008850
x4,0.008850,0.008850,0.008850,0.000000,0.008850,0.008850,0.008850,0.017699,0.026549,0.017699,...,0.008850,0.008850,0.008850,0.000000,0.008850,0.070796,0.061947,0.053097,0.017699,0.000000
x5,0.008850,0.008850,0.008850,0.008850,0.000000,0.008850,0.008850,0.017699,0.026549,0.026549,...,0.017699,0.008850,0.008850,0.008850,0.008850,0.070796,0.070796,0.061947,0.026549,0.008850
x6,0.000000,0.008850,0.008850,0.008850,0.008850,0.000000,0.008850,0.008850,0.017699,0.026549,...,0.017699,0.000000,0.008850,0.008850,0.000000,0.061947,0.070796,0.061947,0.026549,0.008850
x7,0.008850,0.000000,0.000000,0.008850,0.008850,0.008850,0.000000,0.008850,0.017699,0.017699,...,0.008850,0.008850,0.000000,0.008850,0.008850,0.061947,0.061947,0.053097,0.017699,0.008850
x8,0.008850,0.008850,0.008850,0.017699,0.017699,0.008850,0.008850,0.000000,0.008850,0.017699,...,0.017699,0.008850,0.008850,0.017699,0.008850,0.053097,0.061947,0.053097,0.017699,0.017699
x9,0.017699,0.017699,0.017699,0.026549,0.026549,0.017699,0.017699,0.008850,0.000000,0.008850,...,0.026549,0.017699,0.017699,0.026549,0.017699,0.044248,0.053097,0.044248,0.017699,0.026549
x10,0.026549,0.017699,0.017699,0.017699,0.026549,0.026549,0.017699,0.017699,0.008850,0.000000,...,0.017699,0.026549,0.017699,0.017699,0.026549,0.053097,0.044248,0.035398,0.008850,0.017699


In [16]:
def compute_Ci(D, r):
    n_vec = D.shape[0]
    C = np.sum(D <= r, axis=1) / n_vec   # includes self-match (diagonal = 0 <= r)
    return C

C_m = compute_Ci(D_m, r)

C_m_df = pd.DataFrame({
    "Vector": labels_m,
    "Matches (d<=r, incl. self)": np.sum(D_m <= r, axis=1),
    "Ci": C_m
})
C_m_df

,Vector,"Matches (d<=r, incl. self)",Ci
0,x1,5,0.125
1,x2,5,0.125
2,x3,5,0.125
3,x4,3,0.075
4,x5,1,0.025
5,x6,5,0.125
6,x7,5,0.125
7,x8,5,0.125
8,x9,1,0.025
9,x10,1,0.025


In [17]:
phi_m = np.mean(np.log(C_m))
print(f"phi^m = {phi_m:.4f}")

phi^m = -2.6847


## Step 6 — Repeat Steps 2–5 for vector length m+1 = 3

In [18]:
vecs_n = form_vectors(AF, n)
labels_n = [f"Y{i+1}" for i in range(len(vecs_n))]

vec_table_n = pd.DataFrame({
    "Vector": labels_n,
    "Values": [list(v) for v in vecs_n]
})
print(f"Number of vectors = N - (m+1) + 1 = {len(vecs_n)}")
display(vec_table_n)

D_n = distance_matrix(vecs_n)
D_n_df = pd.DataFrame(D_n, index=labels_n, columns=labels_n)
display(D_n_df)

C_n = compute_Ci(D_n, r)
C_n_df = pd.DataFrame({
    "Vector": labels_n,
    "Matches (d<=r, incl. self)": np.sum(D_n <= r, axis=1),
    "Ci": C_n
})
display(C_n_df)

phi_n = np.mean(np.log(C_n))
print(f"phi^(m+1) = {phi_n:.4f}")

Number of vectors = N - (m+1) + 1 = 39


,Vector,Values
0,Y1,"[0.008849557522123894, 0.017699115044247787, 0..."
1,Y2,"[0.017699115044247787, 0.017699115044247787, 0..."
2,Y3,"[0.017699115044247787, 0.017699115044247787, 0..."
3,Y4,"[0.017699115044247787, 0.008849557522123894, 0..."
4,Y5,"[0.008849557522123894, 0.008849557522123894, 0..."
5,Y6,"[0.008849557522123894, 0.017699115044247787, 0..."
6,Y7,"[0.017699115044247787, 0.017699115044247787, 0..."
7,Y8,"[0.017699115044247787, 0.02654867256637168, 0...."
8,Y9,"[0.02654867256637168, 0.035398230088495575, 0...."
9,Y10,"[0.035398230088495575, 0.02654867256637168, 0...."


,Y1,Y2,Y3,Y4,Y5,Y6,Y7,Y8,Y9,Y10,...,Y30,Y31,Y32,Y33,Y34,Y35,Y36,Y37,Y38,Y39
Y1,0.000000,0.008850,0.008850,0.008850,0.008850,0.000000,0.008850,0.017699,0.017699,0.026549,...,0.017699,0.017699,0.000000,0.008850,0.008850,0.061947,0.061947,0.070796,0.061947,0.026549
Y2,0.008850,0.000000,0.008850,0.008850,0.008850,0.008850,0.008850,0.017699,0.017699,0.017699,...,0.008850,0.008850,0.008850,0.008850,0.008850,0.061947,0.061947,0.061947,0.053097,0.017699
Y3,0.008850,0.008850,0.000000,0.008850,0.008850,0.008850,0.017699,0.026549,0.017699,0.017699,...,0.008850,0.008850,0.008850,0.000000,0.008850,0.070796,0.061947,0.061947,0.053097,0.017699
Y4,0.008850,0.008850,0.008850,0.000000,0.008850,0.008850,0.017699,0.026549,0.026549,0.017699,...,0.017699,0.008850,0.008850,0.008850,0.008850,0.070796,0.070796,0.061947,0.053097,0.017699
Y5,0.008850,0.008850,0.008850,0.008850,0.000000,0.008850,0.008850,0.017699,0.026549,0.026549,...,0.017699,0.017699,0.008850,0.008850,0.008850,0.061947,0.070796,0.070796,0.061947,0.026549
Y6,0.000000,0.008850,0.008850,0.008850,0.008850,0.000000,0.008850,0.017699,0.017699,0.026549,...,0.017699,0.017699,0.000000,0.008850,0.008850,0.061947,0.061947,0.070796,0.061947,0.026549
Y7,0.008850,0.008850,0.017699,0.017699,0.008850,0.008850,0.000000,0.008850,0.017699,0.017699,...,0.017699,0.008850,0.008850,0.017699,0.008850,0.053097,0.061947,0.061947,0.053097,0.017699
Y8,0.017699,0.017699,0.026549,0.026549,0.017699,0.017699,0.008850,0.000000,0.008850,0.017699,...,0.026549,0.017699,0.017699,0.026549,0.017699,0.044248,0.053097,0.061947,0.053097,0.026549
Y9,0.017699,0.017699,0.017699,0.026549,0.026549,0.017699,0.017699,0.008850,0.000000,0.008850,...,0.017699,0.026549,0.017699,0.017699,0.026549,0.053097,0.044248,0.053097,0.044248,0.017699
Y10,0.026549,0.017699,0.017699,0.017699,0.026549,0.026549,0.017699,0.017699,0.008850,0.000000,...,0.017699,0.017699,0.026549,0.017699,0.017699,0.053097,0.053097,0.044248,0.035398,0.017699


,Vector,"Matches (d<=r, incl. self)",Ci
0,Y1,3,0.076923
1,Y2,1,0.025641
2,Y3,2,0.051282
3,Y4,1,0.025641
4,Y5,1,0.025641
5,Y6,3,0.076923
6,Y7,2,0.051282
7,Y8,1,0.025641
8,Y9,1,0.025641
9,Y10,1,0.025641


phi^(m+1) = -3.3524


In [19]:
ApEn_manual = phi_m - phi_n
print(f"ApEn = phi^m - phi^(m+1) = {phi_m:.4f} - ({phi_n:.4f}) = {ApEn_manual:.4f}")

ApEn = phi^m - phi^(m+1) = -2.6847 - (-3.3524) = 0.6676


---
# PART 2 — Sample Entropy (manual, worked step-by-step)

SampEn **excludes self-matches** and counts each pair only once (upper-triangle, i < j).

### Step 1 — Reuse length-m vectors, build pairwise (i<j) distance table, count matches B

In [20]:
def pairwise_matches(vecs, labels, r):
    rows = []
    n_vec = len(vecs)
    for i in range(n_vec):
        for j in range(i + 1, n_vec):
            d = np.max(np.abs(vecs[i] - vecs[j]))
            match = d <= r
            rows.append({"Pair": f"{labels[i]}-{labels[j]}", "Distance": d, f"Match (d<={r})": match})
    return pd.DataFrame(rows)

pairs_m = pairwise_matches(vecs_m, labels_m, r)
display(pairs_m)

B = pairs_m.iloc[:, 2].sum()
print(f"Number of matching pairs (length m={m}):  B = {B}")

,Pair,Distance,Match (d<=0.0022092020862376836)
0,x1-x2,0.008850,False
1,x1-x3,0.008850,False
2,x1-x4,0.008850,False
3,x1-x5,0.008850,False
4,x1-x6,0.000000,True
...,...,...,...
775,x37-x39,0.053097,False
776,x37-x40,0.061947,False
777,x38-x39,0.035398,False
778,x38-x40,0.053097,False


Number of matching pairs (length m=2):  B = 47


### Step 2 — Same, for length m+1 = 3 vectors → count matches A

In [21]:
pairs_n = pairwise_matches(vecs_n, labels_n, r)
display(pairs_n)

A = pairs_n.iloc[:, 2].sum()
print(f"Number of matching pairs (length m+1={n}):  A = {A}")

,Pair,Distance,Match (d<=0.0022092020862376836)
0,Y1-Y2,0.008850,False
1,Y1-Y3,0.008850,False
2,Y1-Y4,0.008850,False
3,Y1-Y5,0.008850,False
4,Y1-Y6,0.000000,True
...,...,...,...
736,Y36-Y38,0.053097,False
737,Y36-Y39,0.061947,False
738,Y37-Y38,0.035398,False
739,Y37-Y39,0.053097,False


Number of matching pairs (length m+1=3):  A = 10


### Step 3 — SampEn = -ln(A / B)

In [22]:
SampEn_manual = -np.log(A / B)
print(f"SampEn = -ln(A/B) = -ln({A}/{B}) = {SampEn_manual:.4f}")

SampEn = -ln(A/B) = -ln(10/47) = 1.5476
